# Chapter 5 Lab: Neural Networks and Optimization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from matplotlib.colors import ListedColormap

%matplotlib inline

# Seed for reproducibility
np.random.seed(42)

## Forward Pass by Hand

Here we build a tiny network by hand and compute the forward pass step by step.

Network architecture:
- Input: 2 neurons
- Hidden layer: 2 sigmoid units
- Output: 1 sigmoid unit (binary classification)

In [ ]:
def sigmoid(z):
    """Sigmoid activation function."""
    return 1 / (1 + np.exp(-z))

# Input
x = np.array([1.0, 0.5])
print(f"Input: x = {x}")
print()

# Weights for hidden layer (2 input -> 2 hidden)
W1 = np.array([
    [0.5, -0.3],  # weights to hidden unit 1 and 2 from input 1
    [0.2, 0.8]    # weights to hidden unit 1 and 2 from input 2
])
b1 = np.array([0.1, -0.2])

# Weights for output layer (2 hidden -> 1 output)
W2 = np.array([[0.6], [0.4]])
b2 = np.array([0.0])

# Forward pass: hidden layer
z1 = x @ W1 + b1
print(f"Hidden pre-activations (z1): {z1}")

a1 = sigmoid(z1)
print(f"Hidden activations (a1 = sigmoid(z1)): {a1}")
print()

# Forward pass: output layer
z2 = a1 @ W2 + b2
print(f"Output pre-activation (z2): {z2}")

a2 = sigmoid(z2)
print(f"Output activation (a2 = sigmoid(z2)): {a2}")
print()
print(f"Predicted probability: {a2[0]:.4f}")

## Learning Rate Experiment

We minimize L(w) = (w - 3)^2 using gradient descent with different learning rates.
This demonstrates how learning rate affects convergence and stability.

In [ ]:
def loss_fn(w):
    """Loss function: (w - 3)^2"""
    return (w - 3)**2

def grad_fn(w):
    """Gradient: d/dw (w - 3)^2 = 2(w - 3)"""
    return 2 * (w - 3)

# Test different learning rates
learning_rates = [0.01, 0.1, 0.5, 1.5]
iterations = 50
w_init = 0.0  # starting weight

results = {}

for lr in learning_rates:
    w = w_init
    weights = [w]
    losses = [loss_fn(w)]
    
    for i in range(iterations):
        grad = grad_fn(w)
        w = w - lr * grad
        weights.append(w)
        losses.append(loss_fn(w))
    
    results[lr] = {"weights": weights, "losses": losses}

# Plot weight trajectories
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: Weight vs iteration
for lr in learning_rates:
    weights = results[lr]["weights"]
    axes[0].plot(weights, label=f"lr={lr}", linewidth=2, alpha=0.7)

axes[0].axhline(y=3, color="red", linestyle="--", label="Optimal w=3", linewidth=2)
axes[0].set_xlabel("Iteration", fontsize=11)
axes[0].set_ylabel("Weight (w)", fontsize=11)
axes[0].set_title("Weight Evolution with Different Learning Rates", fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Loss vs iteration
for lr in learning_rates:
    losses = results[lr]["losses"]
    axes[1].semilogy(losses, label=f"lr={lr}", linewidth=2, alpha=0.7)

axes[1].set_xlabel("Iteration", fontsize=11)
axes[1].set_ylabel("Loss (log scale)", fontsize=11)
axes[1].set_title("Loss Decay with Different Learning Rates", fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.show()

print("\nFinal weights after 50 iterations:")
for lr in learning_rates:
    final_w = results[lr]["weights"][-1]
    final_loss = results[lr]["losses"][-1]
    print(f"  lr={lr:3.1f}: w={final_w:8.4f}, loss={final_loss:.6f}")

## MLP vs Linear on make_moons

We compare a nonlinear neural network (MLP) with a linear model (logistic regression)
on the make_moons dataset, which requires nonlinearity to separate.

In [ ]:
# Generate make_moons dataset
X, y = make_moons(n_samples=300, noise=0.1, random_state=42)

# Normalize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train models
mlp = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="sigmoid",
    max_iter=1000,
    random_state=42
)
mlp.fit(X_scaled, y)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_scaled, y)

# Create decision boundary mesh
h = 0.02
x_min, x_max = X_scaled[:, 0].min() - 0.5, X_scaled[:, 0].max() + 0.5
y_min, y_max = X_scaled[:, 1].min() - 0.5, X_scaled[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

# Predictions for mesh
Z_mlp = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_lr = lr.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_light = ListedColormap(["#FFAAAA", "#AAAAFF"])
cmap_bold = ListedColormap(["#FF0000", "#0000FF"])

# MLP plot
axes[0].contourf(xx, yy, Z_mlp, cmap=cmap_light, alpha=0.8)
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap=cmap_bold, edgecolors="k", s=30)
axes[0].set_xlim(xx.min(), xx.max())
axes[0].set_ylim(yy.min(), yy.max())
axes[0].set_xlabel("Feature 1 (normalized)", fontsize=11)
axes[0].set_ylabel("Feature 2 (normalized)", fontsize=11)
axes[0].set_title(f"MLP (50 hidden units, sigmoid) | Accuracy: {mlp.score(X_scaled, y):.3f}", fontsize=12)

# Logistic Regression plot
axes[1].contourf(xx, yy, Z_lr, cmap=cmap_light, alpha=0.8)
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap=cmap_bold, edgecolors="k", s=30)
axes[1].set_xlim(xx.min(), xx.max())
axes[1].set_ylim(yy.min(), yy.max())
axes[1].set_xlabel("Feature 1 (normalized)", fontsize=11)
axes[1].set_ylabel("Feature 2 (normalized)", fontsize=11)
axes[1].set_title(f"Logistic Regression (linear) | Accuracy: {lr.score(X_scaled, y):.3f}", fontsize=12)

plt.tight_layout()
plt.show()

print(f"MLP accuracy:               {mlp.score(X_scaled, y):.4f}")
print(f"Logistic Regression accuracy: {lr.score(X_scaled, y):.4f}")

## Overfitting and Regularization

We train an MLP with and without L2 regularization (alpha parameter) on a small training set
to show how regularization smooths the decision boundary.

In [ ]:
# Generate smaller dataset for overfitting
X_small, y_small = make_moons(n_samples=50, noise=0.15, random_state=42)
X_small_scaled = scaler.fit_transform(X_small)

# Train with no regularization
mlp_no_reg = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="sigmoid",
    alpha=0.0,  # No L2 regularization
    max_iter=1000,
    random_state=42
)
mlp_no_reg.fit(X_small_scaled, y_small)

# Train with L2 regularization
mlp_reg = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="sigmoid",
    alpha=0.1,  # L2 regularization
    max_iter=1000,
    random_state=42
)
mlp_reg.fit(X_small_scaled, y_small)

# Predictions for mesh
Z_no_reg = mlp_no_reg.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
Z_reg = mlp_reg.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# No regularization
axes[0].contourf(xx, yy, Z_no_reg, cmap=cmap_light, alpha=0.8)
axes[0].scatter(X_small_scaled[:, 0], X_small_scaled[:, 1], c=y_small, cmap=cmap_bold, edgecolors="k", s=50)
axes[0].set_xlim(xx.min(), xx.max())
axes[0].set_ylim(yy.min(), yy.max())
axes[0].set_xlabel("Feature 1 (normalized)", fontsize=11)
axes[0].set_ylabel("Feature 2 (normalized)", fontsize=11)
axes[0].set_title(f"No Regularization (alpha=0.0) | Accuracy: {mlp_no_reg.score(X_small_scaled, y_small):.3f}", fontsize=12)

# With regularization
axes[1].contourf(xx, yy, Z_reg, cmap=cmap_light, alpha=0.8)
axes[1].scatter(X_small_scaled[:, 0], X_small_scaled[:, 1], c=y_small, cmap=cmap_bold, edgecolors="k", s=50)
axes[1].set_xlim(xx.min(), xx.max())
axes[1].set_ylim(yy.min(), yy.max())
axes[1].set_xlabel("Feature 1 (normalized)", fontsize=11)
axes[1].set_ylabel("Feature 2 (normalized)", fontsize=11)
axes[1].set_title(f"With Regularization (alpha=0.1) | Accuracy: {mlp_reg.score(X_small_scaled, y_small):.3f}", fontsize=12)

plt.tight_layout()
plt.show()

print(f"No regularization accuracy:   {mlp_no_reg.score(X_small_scaled, y_small):.4f}")
print(f"Regularized (alpha=0.1) accuracy: {mlp_reg.score(X_small_scaled, y_small):.4f}")

## Loss Curves: Training vs Validation

We train an MLP and track both training and validation loss over epochs to identify where overfitting begins.

In [ ]:
# Generate dataset and split
X_full, y_full = make_moons(n_samples=300, noise=0.1, random_state=42)
X_full_scaled = scaler.fit_transform(X_full)

X_train, X_val, y_train, y_val = train_test_split(
    X_full_scaled, y_full, test_size=0.3, random_state=42
)

# Train with partial_fit to record loss at each iteration
mlp_track = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="sigmoid",
    max_iter=1,
    warm_start=True,
    random_state=42
)

train_losses = []
val_losses = []

n_epochs = 200
for epoch in range(n_epochs):
    mlp_track.fit(X_train, y_train)
    
    # Compute log loss (binary cross-entropy)
    train_pred_proba = mlp_track.predict_proba(X_train)
    val_pred_proba = mlp_track.predict_proba(X_val)
    
    # Clip probabilities to avoid log(0)
    eps = 1e-15
    train_pred_proba = np.clip(train_pred_proba, eps, 1 - eps)
    val_pred_proba = np.clip(val_pred_proba, eps, 1 - eps)
    
    train_loss = -np.mean(y_train * np.log(train_pred_proba[:, 1]) + (1 - y_train) * np.log(train_pred_proba[:, 0]))
    val_loss = -np.mean(y_val * np.log(val_pred_proba[:, 1]) + (1 - y_val) * np.log(val_pred_proba[:, 0]))
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(train_losses, label="Training Loss", linewidth=2.5, color="#1f77b4", alpha=0.8)
ax.plot(val_losses, label="Validation Loss", linewidth=2.5, color="#ff7f0e", alpha=0.8)

# Mark where validation loss starts increasing (overfitting point)
best_val_epoch = np.argmin(val_losses)
ax.axvline(best_val_epoch, color="red", linestyle="--", alpha=0.5, label=f"Best epoch: {best_val_epoch}")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss (binary cross-entropy)", fontsize=12)
ax.set_title("Training vs Validation Loss Over Time", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best validation loss: {min(val_losses):.4f} at epoch {best_val_epoch}")
print(f"Final validation loss: {val_losses[-1]:.4f}")
print(f"Overfitting gap (final - best): {val_losses[-1] - min(val_losses):.4f}")

## What to Notice

1. **Forward Pass**: Each layer transforms the input through weights and nonlinear activations. The sigmoid squashes values to (0, 1).

2. **Learning Rate**: Too small (0.01) converges slowly; too large (1.5) diverges. The sweet spot (0.1–0.5) balances speed and stability.

3. **Nonlinearity**: MLPs can learn curved decision boundaries (moons dataset), while linear models cannot. The hidden layer creates the nonlinear capacity.

4. **Regularization**: L2 penalty (alpha > 0) shrinks weights, leading to smoother, simpler boundaries. This reduces overfitting on small datasets.

5. **Training vs Validation**: Training loss always decreases, but validation loss can increase. The gap signals overfitting. Early stopping at the best validation epoch prevents this.